In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for compliance (not used to convert timestamps per instructions)
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')

# Ensure 'value' is numeric
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# Helper to aggregate groups with required percentiles
def agg_stats(df, group_cols, value_col='value'):
    grouped = (
        df.groupby(group_cols)[value_col]
          .agg(count='count',
               min='min',
               max='max',
               mean='mean',
               median=lambda x: x.quantile(0.5),
               p90=lambda x: x.quantile(0.9),
               p95=lambda x: x.quantile(0.95))
          .reset_index()
    )
    return grouped

# 1) metric.csv groups
metric_groups = agg_stats(metric, ['cmdb_id', 'kpi_name'])
metric_top20 = metric_groups.sort_values('count', ascending=False)[
    ['cmdb_id','kpi_name','count','min','mean','median','p90','p95','max']
].head(20)

# 2) trace.csv groups
trace_groups = agg_stats(trace, ['cmdb_id', 'trace_name'])
trace_top20 = trace_groups.sort_values('count', ascending=False)[
    ['cmdb_id','trace_name','count','min','mean','median','p90','p95','max']
].head(20)

# 3) log.csv groups
log_groups = agg_stats(log, ['cmdb_id', 'log_name'])
log_top20 = log_groups.sort_values('count', ascending=False)[
    ['cmdb_id','log_name','count','min','mean','median','p90','p95','max']
].head(20)

# Return compact summaries (top 20 each). Variables metric_groups, trace_groups, log_groups are cached for later steps.
metric_top20, trace_top20, log_top20

```
Out[1]:
```
```python
summary = (
    "Summary of grouped telemetry (top 20 groups each):\n"
    "- Metric groups: all top groups have 31 samples. Key observations: IG01 JVM heap shows "
    "HeapMemoryUsed median ~7.94e8, mean ~4.65e8 and max ~8.82e8 while HeapMemoryMax is ~2.04e9 "
    "(heap usage below configured max). JVM thread count for IG01 is ~73. Several Tomcat session "
    "metrics are zero and some Tomcat thread pools show constant high values (100) for certain pools.\n"
    "- Trace groups: dockerA2/dockerB2 dominate counts. Many trace p95 values for calls to MG01/MG02 "
    "are around ~1s while their means are much lower (tens to hundreds of ms), indicating tail latency on those paths. "
    "trace.row_count is large for dockerA nodes (hundreds to thousands), meaning these services handle high request volumes.\n"
    "- Log groups: Tomcat and Apache log.row_count are in the hundreds per interval (300–800), while log.error_count is "
    "essentially zero for top hosts. Tomcat gc_pause_max values are small (~0.05–0.07s).\n\n"
    "Overall conclusion: there are no obvious sustained error spikes or JVM OOM signs in the top summaries. "
    "The most actionable signal is trace tail latency (p95 ~1s) on calls to MG01/MG02 from dockerA/dockerB — investigate those call paths and endpoints. "
    "Also continue to monitor IG01 JVM heap usage (currently below max) and Tomcat thread pool configurations."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id                                           kpi_name  count           min          mean        median           p90           p95           max
1373      Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  0.000000e+00  1.967742e+00  2.000000e+00  4.000000e+00  4.000000e+00  5.000000e+00
6             IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  6.900000e+01  7.290323e+01  7.300000e+01  7.500000e+01  7.550000e+01  7.600000e+01
3             IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.848308e+07  5.848308e+07  5.848308e+07  5.848308e+07  5.848308e+07  5.848308e+07
1372      Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
2             IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  6.442168e+07  4.649326e+08  4.691545e+08  7.940797e+08  8.312444e+08  8.818901e+08
0             IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09
1371      Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1370      Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1369      Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
300           MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00  9.677419e-02  0.000000e+00  0.000000e+00  5.000000e-01  2.000000e+00
878        Redis02                         OSLinux-CPU_CPU_CPUSysTime     31  7.186130e-01  1.033002e+00  1.055400e+00  1.126654e+00  1.171050e+00  1.241246e+00
879        Redis02                        OSLinux-CPU_CPU_CPUUserTime     31  2.841000e-01  3.743630e-01  3.727000e-01  4.230000e-01  4.234500e-01  4.979000e-01
1189      Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1188      Tomcat02     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1281      Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1280      Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1020  ServiceTest8                                             app.sr     31  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
1019  ServiceTest8                                             app.rr     31  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
1195      Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
1184      Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00,       cmdb_id                     trace_name  count        min         mean       median          p90          p95          max
260  dockerB2        trace.to_MG01.row_count     31   1.000000    59.419355    58.000000    98.000000   103.500000   107.000000
259  dockerB2     trace.to_MG01.duration_p95     31   0.006000     0.314761     0.190000     0.969550     1.013850     1.017000
258  dockerB2    trace.to_MG01.duration_mean     31   0.006000     0.055449     0.051951     0.080609     0.098301     0.182966
219  dockerA2       trace.self.duration_mean     31   0.008097     0.014562     0.013755     0.019025     0.020231     0.034753
220  dockerA2        trace.self.duration_p95     31   0.001000     0.004726     0.006000     0.008000     0.008500     0.010000
221  dockerA2           trace.self.row_count     31  89.000000  2815.000000  2597.000000  3678.000000  3823.500000  4208.000000
222  dockerA2    trace.to_MG01.duration_mean     31   0.006000     0.062616     0.045340     0.105536     0.195544     0.259750
223  dockerA2     trace.to_MG01.duration_p95     31   0.006850     0.287960     0.042200     1.016000     1.021950     1.088000
224  dockerA2        trace.to_MG01.row_count     31   1.000000    43.161290    42.000000    71.000000    76.500000    81.000000
225  dockerA2    trace.to_MG02.duration_mean     31   0.006667     0.044968     0.047077     0.089892     0.098109     0.120821
226  dockerA2     trace.to_MG02.duration_p95     31   0.007900     0.243210     0.061450     0.770750     1.015775     1.018400
211  dockerA2   trace.from_MG01.duration_p95     31   0.049500     0.912181     1.026000     1.030000     1.031800     1.172200
212  dockerA2      trace.from_MG01.row_count     31   7.000000   207.064516   199.000000   275.000000   293.000000   311.000000
213  dockerA2  trace.from_MG02.duration_mean     31   0.066000     0.116681     0.115948     0.145829     0.176471     0.192675
214  dockerA2   trace.from_MG02.duration_p95     31   0.195700     0.919298     1.025000     1.028000     1.031000     1.033800
215  dockerA2      trace.from_MG02.row_count     31   6.000000   203.451613   202.000000   251.000000   269.500000   283.000000
203  dockerA1           trace.self.row_count     31  52.000000   927.870968   884.000000  1176.000000  1250.500000  1468.000000
210  dockerA2  trace.from_MG01.duration_mean     31   0.056327     0.115765     0.107448     0.142415     0.164671     0.336286
195  dockerA1  trace.from_MG02.duration_mean     31   0.050492     0.147982     0.129943     0.219843     0.230248     0.427667
196  dockerA1   trace.from_MG02.duration_p95     31   0.097000     0.922955     1.028000     1.039000     1.039875     1.044000,      cmdb_id          log_name  count    min        mean  median      p90      p95     max
9   Tomcat02     log.row_count     31   7.00  346.354839  371.00  397.000  406.000  418.00
7   Tomcat02   log.error_count     31   0.00    0.000000    0.00    0.000    0.000    0.00
6   Tomcat01     log.row_count     31   6.00  348.161290  357.00  392.000  406.500  429.00
4   Tomcat01   log.error_count     31   0.00    0.000000    0.00    0.000    0.000    0.00
18  apache02   log.error_count     31   0.00    0.000000    0.00    0.000    0.000    0.00
17  apache01     log.row_count     31  14.00  696.548387  734.00  787.000  815.500  841.00
19  apache02     log.row_count     31  18.00  695.483871  722.00  790.000  832.000  850.00
16  apache01   log.error_count     31   0.00    0.000000    0.00    0.000    0.000    0.00
10  Tomcat03   log.error_count     31   0.00    0.000000    0.00    0.000    0.000    0.00
15  Tomcat04     log.row_count     31   6.00  349.870968  357.00  419.000  421.000  439.00
13  Tomcat04   log.error_count     31   0.00    0.000000    0.00    0.000    0.000    0.00
12  Tomcat03     log.row_count     31   9.00  349.709677  363.00  408.000  421.000  430.00
1    Mysql01     log.row_count     30   1.00    5.600000    5.50    9.000    9.000   14.00
0    Mysql01   log.error_count     30   0.00    0.000000    0.00    0.000    0.000    0.00
14  Tomcat04  log.gc_pause_max     19   0.02    0.053684    0.05    0.080    0.082    0.10
11  Tomcat03  log.gc_pause_max     17   0.02    0.055294    0.07    0.080    0.082    0.09
5   Tomcat01  log.gc_pause_max     17   0.02    0.054706    0.06    0.084    0.092    0.10
8   Tomcat02  log.gc_pause_max     17   0.04    0.066471    0.07    0.094    0.100    0.10
3    Mysql02     log.row_count      1   1.00    1.000000    1.00    1.000    1.000    1.00
2    Mysql02   log.error_count      1   0.00    0.000000    0.00    0.000    0.000    0.00)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
from datetime import datetime
import pytz

# reuse existing 'metric' DataFrame loaded earlier

# 1) Compute global thresholds (P95 and P5) per (cmdb_id, kpi_name) across full series
metric_thresh_grp = metric.groupby(['cmdb_id', 'kpi_name'])['value']
metric_thresholds = (
    metric_thresh_grp.agg(
        p95=lambda x: x.quantile(0.95),
        p5 = lambda x: x.quantile(0.05),
        full_count='count'
    )
    .reset_index()
)

# 2) Filter metric.csv to incident window (UTC+8): March 24, 2021 12:30 to 13:00 (inclusive)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2021, 3, 24, 12, 30, 0))
end_local   = tz.localize(datetime(2021, 3, 24, 13,  0, 0))
# convert to UTC epoch seconds (timestamps in metric are epoch seconds)
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

metric_filtered = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# join each filtered row with its group's global p95 and p5
metric_filtered = metric_filtered.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')

# 3) Identify anomaly rows where value >= p95 OR value <= p5
# Keep all columns plus p95 and p5 (they are already merged). Store full anomalies.
cond_high = metric_filtered['value'] >= metric_filtered['p95']
cond_low  = metric_filtered['value'] <= metric_filtered['p5']
metric_anomalies = metric_filtered[cond_high | cond_low].copy()

# 4) Aggregate metric_anomalies by (cmdb_id, kpi_name)
if not metric_anomalies.empty:
    agg = (
        metric_anomalies.groupby(['cmdb_id','kpi_name']).agg(
            anomaly_count = ('value','count'),
            earliest_timestamp = ('timestamp','min'),
            latest_timestamp   = ('timestamp','max'),
            min_value = ('value','min'),
            max_value = ('value','max'),
            p95 = ('p95','first'),
            p5  = ('p5','first')
        ).reset_index()
    )
    # severity_ratio = max_value / p95 if p95>0 else NaN
    agg['severity_ratio'] = np.where(agg['p95']>0, agg['max_value'] / agg['p95'], np.nan)

    # sort by severity_ratio descending, placing NaNs at the end
    agg_sorted = agg.assign(_isnull = agg['severity_ratio'].isnull()).sort_values(['_isnull','severity_ratio'], ascending=[True, False]).drop(columns=['_isnull'])
else:
    # empty aggregation
    agg_sorted = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_timestamp','latest_timestamp','min_value','max_value','p95','p5','severity_ratio'])

# Keep variables metric_thresholds and metric_anomalies in kernel for later use.
# Return compact summary: shapes of thresholds/anomalies and top 20 aggregated anomaly groups
metric_thresholds.shape, metric_anomalies.shape, agg_sorted[[
    'cmdb_id','kpi_name','anomaly_count','earliest_timestamp','latest_timestamp','min_value','max_value','p95','p5','severity_ratio'
]].head(20)

```
Out[2]:
```
Summary (using full metric series for thresholds; window = Mar 24, 2021 12:30–13:00 UTC+8):

- Global groups computed: 1,554 (metric_thresholds).
- Rows in the incident window that were flagged as anomalies after joining global thresholds: 17,835 (metric_anomalies). This is a large number of out-of-threshold points inside the 30-minute window.

Top aggregated anomaly groups (top entries by severity_ratio, compact view):
1. MG02 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
   - anomaly_count: 31, earliest/latest timestamps within the window
   - min=0.0, max=2.0, p95=0.5, p5=0.0, severity_ratio=4.0
2. IG01 — OSLinux-CPU_CPU_CPUWio
   - anomaly_count: 4, min=0.0208, max=2.3661, p95≈0.67946, p5≈0.0208, severity_ratio≈3.48
3. MG01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead
   - anomaly_count: 15, min=0.0, max=1.7333, p95=0.52, p5=0.0, severity_ratio≈3.33
4. MG01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps
   - anomaly_count: 15, min=0.0, max=0.2167, p95=0.065, p5=0.0, severity_ratio≈3.33
5. Tomcat01 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps
   - anomaly_count: 15, min=0.0, max=2.0, p95=0.6, p5=0.0, severity_ratio≈3.33
6. Tomcat04 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps
   - anomaly_count: 14, min=0.0, max=1.0, p95=0.35, p5=0.0, severity_ratio≈2.86
7. MG02 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite
   - anomaly_count: 2, min=7.3333, max=12569.07, p95≈4446.58, p5≈7.7667, severity_ratio≈2.83
8. Tomcat01 — OSLinux-CPU_CPU_CPUWio
   - anomaly_count: 4, min≈0.01248, max≈0.87730, p95≈0.31691, p5≈0.01248, severity_ratio≈2.77
9. MG02 — OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps
   - anomaly_count: 2, min≈0.3667, max≈18.1333, p95≈7.17, p5≈0.3883, severity_ratio≈2.53
10. Redis02 — OSLinux-CPU_CPU_CPUWio
    - anomaly_count: 4, min=0.0, max≈0.14258, p95≈0.05655, p5≈0.00418, severity_ratio≈2.52
(Other top groups include Tomcat thread metrics, various disk I/O and CPU idle metrics; the full top-20 table is stored in the kernel variable.)

Interpretation and notes:
- The highest severity ratios are driven by values exceeding small p95 baselines (some p95 values are small), or by very large spikes (e.g., MG02 DSKWrite max ≈12,569).
- Several groups have p5 = 0.0 (so low-end anomalies also occur when values are zero).
- Key hotspots to investigate first:
  - MG02 network (TCP FIN-WAIT) and MG01 disk I/O (DSKRead / DSKRTps) — repeated anomalies and high severity ratios.
  - Tomcat hosts showing local-disk activity spikes (sdb DSKBps) and CPU WIO anomalies.
  - The extreme MG02 DSKWrite spike should be inspected (very large max value).
- Next steps recommended: inspect raw time series for these groups, correlate with traces/logs around the same timestamps, and check whether these spikes correspond to legitimate load, transient spikes, or underlying resource problems (disk subsystem or network issues).

Stored variables for follow-up: metric_thresholds (full group thresholds) and metric_anomalies (all flagged rows for the window).

The original code execution output of IPython Kernel is also provided below for reference:

((1554, 5), (17835, 7),        cmdb_id                                           kpi_name  anomaly_count  earliest_timestamp  latest_timestamp  min_value     max_value          p95        p5  severity_ratio
300       MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31          1616560200        1616562000   0.000000      2.000000     0.500000  0.000000        4.000000
15        IG01                             OSLinux-CPU_CPU_CPUWio              4          1616560860        1616562000   0.020800      2.366100     0.679460  0.020800        3.482324
211       MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             15          1616560200        1616561940   0.000000      1.733333     0.520000  0.000000        3.333333
210       MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps             15          1616560200        1616561940   0.000000      0.216667     0.065000  0.000000        3.333333
1061  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps             15          1616560200        1616561940   0.000000      2.000000     0.600000  0.000000        3.333333
1322  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps             14          1616560260        1616561820   0.000000      1.000000     0.350000  0.000000        2.857143
296       MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite              2          1616560200        1616560620   7.333333  12569.066667  4446.580000  7.766667        2.826682
1037  Tomcat01                             OSLinux-CPU_CPU_CPUWio              4          1616560380        1616561880   0.012476      0.877302     0.316911  0.012477        2.768294
295       MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps              2          1616560200        1616560620   0.366667     18.133333     7.170000  0.388333        2.529056
880    Redis02                             OSLinux-CPU_CPU_CPUWio              4          1616560380        1616561700   0.000000      0.142582     0.056550  0.004183        2.521344
1284  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...              8          1616560200        1616561760   0.000000      8.000000     3.500000  0.000000        2.285714
1137  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             13          1616560200        1616561940   0.000000      5.000000     2.200000  0.000000        2.272727
258       MG02                    OSLinux-CPU_CPU-2_SingleCpuidle             13          1616560200        1616561820   0.000000     96.057884    45.299947  0.000000        2.120486
48        IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps             11          1616560320        1616562000   0.000000      1.000000     0.500000  0.000000        2.000000
436    Mysql01                    Mysql-MySQL_3306_ThreadsRunning             29          1616560260        1616561940   1.000000      2.000000     1.000000  1.000000        2.000000
1052  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             15          1616560200        1616562000   0.000000      3.000000     1.500000  0.000000        2.000000
101       IG02                             OSLinux-CPU_CPU_CPUWio              2          1616560440        1616561460   0.016600      0.794300     0.397490  0.020380        1.998289
1227  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps             10          1616560440        1616561820   0.000000      0.750000     0.388333  0.000000        1.931332
496    Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...              2          1616560200        1616561520   7.000000    486.000000   262.500000  7.500000        1.851429
227       MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             30          1616560200        1616562000   0.000000      1.000000     0.550000  0.000000        1.818182)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
from datetime import datetime
import pytz

# Reuse existing 'trace' DataFrame; ensure 'value' is numeric
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) across the full series
trace_thresh_grp = trace.groupby(['cmdb_id', 'trace_name'])['value']
trace_thresholds = (
    trace_thresh_grp.agg(
        p95=lambda x: x.quantile(0.95),
        p5 = lambda x: x.quantile(0.05),
        full_count='count'
    )
    .reset_index()
)

# 2) Filter trace to incident window (UTC+8): March 24, 2021 12:30 to 13:00 (inclusive)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2021, 3, 24, 12, 30, 0))
end_local   = tz.localize(datetime(2021, 3, 24, 13,  0, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

trace_filtered = trace[(trace['timestamp'] >= start_ts) & (trace['timestamp'] <= end_ts)].copy()

# Join each filtered row with its group's global p95 and p5
trace_filtered = trace_filtered.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# 3) Identify anomaly rows where value >= p95 OR value <= p5
cond_high = trace_filtered['value'] >= trace_filtered['p95']
cond_low  = trace_filtered['value'] <= trace_filtered['p5']
trace_anomalies = trace_filtered[cond_high | cond_low].copy()

# 4) Aggregate trace_anomalies by (cmdb_id, trace_name)
if not trace_anomalies.empty:
    agg = (
        trace_anomalies.groupby(['cmdb_id','trace_name']).agg(
            anomaly_count = ('value','count'),
            earliest_timestamp = ('timestamp','min'),
            latest_timestamp   = ('timestamp','max'),
            min_value = ('value','min'),
            max_value = ('value','max'),
            p95 = ('p95','first'),
            p5  = ('p5','first')
        ).reset_index()
    )
    agg['severity_ratio'] = np.where(agg['p95']>0, agg['max_value'] / agg['p95'], np.nan)
    # Sort by severity_ratio desc, placing NaNs at the end
    agg_sorted = agg.assign(_isnull = agg['severity_ratio'].isnull()).sort_values(['_isnull','severity_ratio'], ascending=[True, False]).drop(columns=['_isnull'])
else:
    agg_sorted = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_timestamp','latest_timestamp','min_value','max_value','p95','p5','severity_ratio'])

# Keep trace_thresholds and trace_anomalies in kernel for later correlation.
# Return compact summary: shapes and top 20 aggregated anomaly groups (selected columns)
trace_thresholds.shape, trace_anomalies.shape, agg_sorted[[
    'cmdb_id','trace_name','anomaly_count','earliest_timestamp','latest_timestamp','min_value','max_value','p95','p5','severity_ratio'
]].head(20)

```
Out[3]:
```
```python
summary = (
    "Trace anomaly summary (full-series thresholds; window Mar 24 2021 12:30–13:00 UTC+8):\n"
    "- Computed trace thresholds for 264 (cmdb_id, trace_name) groups; 1,018 trace rows in the window were flagged as anomalies.\n"
    "- Highest severity signals are tail-duration (trace.self.duration_p95) spikes: dockerA1 has the largest severity_ratio (~18.95), "
    "followed by Tomcat01, Tomcat03, and Tomcat04 (severity_ratios ~7.7, ~7.09, ~4.7). These indicate large relative p95 spikes vs historical baselines.\n"
    "- Several cross-service paths show elevated severity (severity_ratio >2): MG02 <-> dockerB2 and MG01 <-> dockerA2/from_dockerB2; many to/from MG01/MG02 duration_mean entries are >2x their p95 baseline.\n"
    "- Interpretation: the dominant issue is tail latency (p95) increases on specific services and inter-service calls, not just uniform latency increase. "
    "dockerA1 and the Tomcat nodes' p95 durations deserve immediate attention, plus MG01/MG02 related RPC paths.\n"
    "- Recommended next steps: inspect raw trace samples and spans for the listed groups/timestamps, correlate these trace spikes with metric anomalies and logs (disk, network, CPU WIO), and prioritize investigation on dockerA1, Tomcat01/03/04, MG01, and MG02 call paths."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((264, 5), (1018, 7),       cmdb_id                         trace_name  anomaly_count  earliest_timestamp  latest_timestamp  min_value  max_value       p95        p5  severity_ratio
202  dockerA1            trace.self.duration_p95              6          1616560260        1616562000   0.001000   0.189500  0.010000  0.001000       18.950000
130  Tomcat01            trace.self.duration_p95              4          1616560200        1616562000   0.022000   0.521000  0.067750  0.023500        7.690037
166  Tomcat03            trace.self.duration_p95              4          1616560200        1616561580   0.022000   1.003000  0.141400  0.023400        7.093352
184  Tomcat04            trace.self.duration_p95              5          1616560200        1616562000   0.022000   0.887550  0.188975  0.023000        4.696653
117      MG02    trace.to_dockerB2.duration_mean              4          1616560200        1616562000   0.059393   2.091000  0.775405  0.097933        2.696654
249  dockerB2      trace.from_MG02.duration_mean              4          1616560200        1616562000   0.059393   2.091000  0.775405  0.097933        2.696654
148  Tomcat02            trace.self.duration_p95              9          1616560200        1616562000   0.024000   0.433000  0.168375  0.025000        2.571641
87       MG02  trace.from_Tomcat04.duration_mean              4          1616560200        1616562000   0.065418   0.800500  0.321282  0.093427        2.491581
189  Tomcat04        trace.to_MG02.duration_mean              4          1616560200        1616562000   0.065418   0.800500  0.321282  0.093427        2.491581
69       MG01    trace.to_dockerA2.duration_mean              4          1616560260        1616562000   0.056327   0.336286  0.164671  0.074712        2.042163
210  dockerA2      trace.from_MG01.duration_mean              4          1616560260        1616562000   0.056327   0.336286  0.164671  0.074712        2.042163
57       MG01  trace.from_dockerB2.duration_mean              4          1616560500        1616562000   0.006000   0.182966  0.098301  0.008360        1.861283
258  dockerB2        trace.to_MG01.duration_mean              4          1616560500        1616562000   0.006000   0.182966  0.098301  0.008360        1.861283
108      MG02    trace.to_dockerA1.duration_mean              4          1616560200        1616562000   0.050492   0.427667  0.230248  0.072671        1.857415
195  dockerA1      trace.from_MG02.duration_mean              4          1616560200        1616562000   0.050492   0.427667  0.230248  0.072671        1.857415
219  dockerA2           trace.self.duration_mean              4          1616560260        1616562000   0.008097   0.034753  0.020231  0.009496        1.717767
39       MG01  trace.from_Tomcat02.duration_mean              4          1616561160        1616562000   0.062203   0.448667  0.268042  0.067775        1.673868
150  Tomcat02        trace.to_MG01.duration_mean              4          1616561160        1616562000   0.062203   0.448667  0.268042  0.067775        1.673868
15       IG01    trace.to_Tomcat04.duration_mean              4          1616560200        1616562000   0.100323   0.806500  0.493041  0.202084        1.635767
174  Tomcat04      trace.from_IG01.duration_mean              4          1616560200        1616562000   0.100323   0.806500  0.493041  0.202084        1.635767)```
```

In [ ]:
# In[4]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[4]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd

# Reuse cached variables: metric_anomalies, metric_thresholds
# Aggregate metric_anomalies by (cmdb_id, kpi_name)
agg = (
    metric_anomalies
      .groupby(['cmdb_id','kpi_name'])['value']
      .agg(
          anomaly_count='count',
          earliest_timestamp='min',
          latest_timestamp='max',
          min_value='min',
          max_value='max'
      )
      .reset_index()
)

# Join p95 and p5 from metric_thresholds to ensure thresholds come from full series
metric_top_anomaly_groups = agg.merge(
    metric_thresholds[['cmdb_id','kpi_name','p95','p5']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Compute severity_ratio = max_value / p95 if p95>0 else NaN
metric_top_anomaly_groups['severity_ratio'] = np.where(
    metric_top_anomaly_groups['p95'] > 0,
    metric_top_anomaly_groups['max_value'] / metric_top_anomaly_groups['p95'],
    np.nan
)

# Sort by severity_ratio desc, placing NaNs at the end
metric_top_anomaly_groups = metric_top_anomaly_groups.sort_values(
    by='severity_ratio', ascending=False, na_position='last'
).reset_index(drop=True)

# Prepare top 10 compact view with required columns
metric_top_anomaly_groups_top10 = metric_top_anomaly_groups[[
    'cmdb_id','kpi_name','anomaly_count','earliest_timestamp','latest_timestamp',
    'min_value','max_value','p95','p5','severity_ratio'
]].head(10)

# Expose the full aggregated table (metric_top_anomaly_groups) for later use and return the top10 summary
metric_top_anomaly_groups.shape, metric_top_anomaly_groups_top10

```
Out[5]:
```
```python
summary = (
    "Top metric anomaly groups (top 10 by severity_ratio):\n"
    "- MG02 | TCP-FIN-WAIT: 31 anomalies, max=2.0, p95=0.5, severity=4.0 — sustained FIN-WAIT activity across the window.\n"
    "- IG01 | CPU WIO: 4 anomalies, max≈2.366, p95≈0.679, severity≈3.48 — high I/O wait spikes on IG01.\n"
    "- MG01 | sda DSKRead: 15 anomalies, max≈1.733, p95=0.52, severity≈3.33 — repeated elevated disk read activity on MG01.\n"
    "- MG01 | sda DSKRTps: 15 anomalies, max≈0.2167, p95=0.065, severity≈3.33 — disk ops rate spike on MG01.\n"
    "- Tomcat01 | sdb DSKBps: 15 anomalies, max=2.0, p95=0.6, severity≈3.33 — Tomcat01 disk throughput spikes.\n"
    "- Tomcat04 | sdb DSKBps: 14 anomalies, max=1.0, p95=0.35, severity≈2.86 — Tomcat04 disk throughput elevated.\n"
    "- MG02 | sdb DSKWrite: 2 anomalies, max≈12,569, p95≈4,446.6, severity≈2.83 — an extreme write spike on MG02 (investigate immediately).\n"
    "- Tomcat01 | CPU WIO: 4 anomalies, max≈0.8773, p95≈0.3169, severity≈2.77 — Tomcat01 I/O wait elevated.\n"
    "- MG02 | sdb DSKWTps: 2 anomalies, max≈18.13, p95≈7.17, severity≈2.53 — high disk write rate bursts on MG02.\n"
    "- Redis02 | CPU WIO: 4 anomalies, max≈0.1426, p95≈0.0566, severity≈2.52 — Redis node showing I/O wait spikes.\n\n"
    "Actionable summary:\n"
    "1) Prioritize MG02 and MG01 (network TCP FIN-WAIT, extreme DSKWrite spike, and disk read/write activity) — these show the highest impact and/or extreme values.\n"
    "2) Investigate I/O wait on IG01, Tomcat01, and Redis02 (could indicate disk or storage contention).\n"
    "3) Correlate these metric groups with trace anomalies and logs at the matching timestamps to determine root cause (disk subsystem, heavy writes, or network issues).\n"
    "4) Stored variables for follow-up: metric_top_anomaly_groups (aggregated) and metric_anomalies (all anomaly rows) are available in the kernel.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((1554, 10),     cmdb_id                                          kpi_name  anomaly_count  earliest_timestamp  latest_timestamp  min_value     max_value          p95        p5  severity_ratio
0      MG02      OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31            0.000000          2.000000   0.000000      2.000000     0.500000  0.000000        4.000000
1      IG01                            OSLinux-CPU_CPU_CPUWio              4            0.020800          2.366100   0.020800      2.366100     0.679460  0.020800        3.482324
2      MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             15            0.000000          1.733333   0.000000      1.733333     0.520000  0.000000        3.333333
3      MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps             15            0.000000          0.216667   0.000000      0.216667     0.065000  0.000000        3.333333
4  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps             15            0.000000          2.000000   0.000000      2.000000     0.600000  0.000000        3.333333
5  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps             14            0.000000          1.000000   0.000000      1.000000     0.350000  0.000000        2.857143
6      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite              2            7.333333      12569.066667   7.333333  12569.066667  4446.580000  7.766667        2.826682
7  Tomcat01                            OSLinux-CPU_CPU_CPUWio              4            0.012476          0.877302   0.012476      0.877302     0.316911  0.012477        2.768294
8      MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps              2            0.366667         18.133333   0.366667     18.133333     7.170000  0.388333        2.529056
9   Redis02                            OSLinux-CPU_CPU_CPUWio              4            0.000000          0.142582   0.000000      0.142582     0.056550  0.004183        2.521344)```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd

# Reuse cached variables: metric_anomalies, metric_top_anomaly_groups, metric_thresholds

# 1) Determine top 10 groups by severity_ratio from metric_top_anomaly_groups (NaNs last)
top_groups = (
    metric_top_anomaly_groups
      .sort_values(by='severity_ratio', ascending=False, na_position='last')
      .head(10)[['cmdb_id','kpi_name']]
)
top_keys = set(tuple(x) for x in top_groups.values)

# 2) Filter metric_anomalies to only these top groups
ma_top = metric_anomalies[
    metric_anomalies.set_index(['cmdb_id','kpi_name']).index.isin(top_keys)
].copy()

# Ensure sorting for segment detection
ma_top.sort_values(['cmdb_id','kpi_name','timestamp'], inplace=True)

# 2) Identify consecutive anomaly segments where adjacent timestamps differ by <= 60s
# Mark new segment when group changes or time gap > 60
grp_change = (ma_top['cmdb_id'] != ma_top['cmdb_id'].shift()) | (ma_top['kpi_name'] != ma_top['kpi_name'].shift())
time_diff = ma_top['timestamp'] - ma_top['timestamp'].shift()
gap_flag = (time_diff > 60) | time_diff.isna()
new_segment = grp_change | gap_flag

# Create a segment id (cumulative sum of new_segment)
ma_top['segment_id'] = new_segment.cumsum()

# 3) Aggregate per (cmdb_id, kpi_name, segment_id)
segments = (
    ma_top.groupby(['cmdb_id','kpi_name','segment_id']).agg(
        start_timestamp = ('timestamp','min'),
        end_timestamp   = ('timestamp','max'),
        length_in_minutes = ('timestamp','count'),
        min_value = ('value','min'),
        max_value = ('value','max')
    ).reset_index(drop=False)
)

# Join p95 from metric_thresholds
segments = segments.merge(
    metric_thresholds[['cmdb_id','kpi_name','p95']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Compute severity_ratio
segments['severity_ratio'] = np.where(segments['p95']>0, segments['max_value'] / segments['p95'], np.nan)

# 4) Exclude segments that are single-point (length_in_minutes == 1) AND severity_ratio < 2.0 (also exclude if severity_ratio is null)
is_single = segments['length_in_minutes'] == 1
is_low_sev_or_null = (segments['severity_ratio'] < 2.0) | (segments['severity_ratio'].isnull())
exclude_mask = is_single & is_low_sev_or_null
metric_faults = segments[~exclude_mask].copy()  # store full faults table in kernel

# 5) Return top 20 fault segments sorted by severity_ratio desc (nulls last)
metric_faults_sorted = metric_faults.assign(_isnull = metric_faults['severity_ratio'].isnull()) \
    .sort_values(['_isnull','severity_ratio'], ascending=[True, False]).drop(columns=['_isnull'])

metric_faults_top20 = metric_faults_sorted[[
    'cmdb_id','kpi_name','start_timestamp','end_timestamp','length_in_minutes',
    'min_value','max_value','p95','severity_ratio'
]].head(20)

metric_faults_top20

```
Out[6]:
```
Summary of detected fault segments (top results):

- MG02 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
  - Sustained segment from 1616560200 to 1616562000 (31 minutes)
  - min=0.0, max=2.0, p95=0.5, severity_ratio=4.00
  - Interpretation: prolonged elevated FIN-WAIT state across the whole window — high priority.

- MG02 | OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite
  - Single-point extreme spike at 1616560620 (length 1)
  - min=max≈12569.07, p95≈4446.58, severity_ratio≈2.83
  - Interpretation: very large write spike at one timestamp — investigate immediately.

- IG01 | OSLinux-CPU_CPU_CPUWio
  - Single-point at 1616562000 (length 1)
  - max≈2.3661, p95≈0.67946, severity_ratio≈3.48
  - Interpretation: I/O-wait spike on IG01 — investigate disk/subsystem contention.

- MG01 | sda DSKRead and sda DSKRTps
  - Each a single-point at 1616561340 with severity_ratio≈3.33
  - Interpretation: short but significant disk read/Rtps spikes on MG01.

- Tomcat01 | sdb DSKBps and Tomcat04 | sdb DSKBps
  - Single-point spikes (severity ≈3.33 and ≈2.86 respectively)
  - Interpretation: disk throughput bursts on Tomcat hosts.

- Tomcat01 | OSLinux-CPU_CPU_CPUWio and Redis02 | OSLinux-CPU_CPU_CPUWio
  - Single-point I/O-wait spikes retained (severity ≈2.77 and ≈2.52)

Notes and recommendations:
- The only multi-minute fault is the MG02 TCP FIN-WAIT condition (31 min) — this is the most persistent issue and highest-priority.
- The MG02 DSKWrite extreme spike is the largest absolute anomaly and also high priority.
- Several single-minute spikes with severity_ratio >= 2 were retained as likely meaningful (disk I/O and I/O-wait on MG01, Tomcat nodes, IG01, Redis02).
- Next steps: correlate these fault segments with traces and logs at the listed timestamps (metric_faults is stored in the kernel), check disk subsystem and network state on MG01/MG02 and Tomcat/IG01/Redis02, and inspect connection behavior causing prolonged FIN-WAIT on MG02.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                          kpi_name  start_timestamp  end_timestamp  length_in_minutes     min_value     max_value          p95  severity_ratio
38      MG02      OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT       1616560200     1616562000                 31      0.000000      2.000000     0.500000        4.000000
3       IG01                            OSLinux-CPU_CPU_CPUWio       1616562000     1616562000                  1      2.366100      2.366100     0.679460        3.482324
28      MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead       1616561340     1616561340                  1      1.733333      1.733333     0.520000        3.333333
13      MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps       1616561340     1616561340                  1      0.216667      0.216667     0.065000        3.333333
61  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps       1616561940     1616561940                  1      2.000000      2.000000     0.600000        3.333333
72  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps       1616561460     1616561460                  1      1.000000      1.000000     0.350000        2.857143
37      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite       1616560620     1616560620                  1  12569.066667  12569.066667  4446.580000        2.826682
46  Tomcat01                            OSLinux-CPU_CPU_CPUWio       1616561880     1616561880                  1      0.877302      0.877302     0.316911        2.768294
35      MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps       1616560620     1616560620                  1     18.133333     18.133333     7.170000        2.529056
39   Redis02                            OSLinux-CPU_CPU_CPUWio       1616560380     1616560380                  1      0.142582      0.142582     0.056550        2.521344```
```

In [ ]:
# In[7]:

import numpy as np
import pandas as pd

# Reuse cached variables: trace_anomalies, trace_thresholds

# 1) Filter trace_anomalies to MG02
mg02_trace_anom = trace_anomalies[trace_anomalies['cmdb_id'] == 'MG02'].copy()

# 2) Aggregate by trace_name
if not mg02_trace_anom.empty:
    mg02_agg = (
        mg02_trace_anom.groupby('trace_name')['value']
        .agg(
            anomaly_count='count',
            earliest_timestamp='min',
            latest_timestamp='max',
            min_value='min',
            max_value='max'
        )
        .reset_index()
    )
    # Join global thresholds (p95, p5) from trace_thresholds
    mg02_agg = mg02_agg.merge(
        trace_thresholds[trace_thresholds['cmdb_id']=='MG02'][['trace_name','p95','p5']],
        on='trace_name',
        how='left'
    )
    # Compute severity_ratio
    mg02_agg['severity_ratio'] = np.where(mg02_agg['p95']>0, mg02_agg['max_value'] / mg02_agg['p95'], np.nan)

    # Sort by severity_ratio desc, place NaNs at the end
    mg02_agg = mg02_agg.assign(_isnull = mg02_agg['severity_ratio'].isnull()) \
                       .sort_values(['_isnull','severity_ratio'], ascending=[True, False]) \
                       .drop(columns=['_isnull']).reset_index(drop=True)
else:
    mg02_agg = pd.DataFrame(columns=['trace_name','anomaly_count','earliest_timestamp','latest_timestamp','min_value','max_value','p95','p5','severity_ratio'])

# Limit to 20 rows for display
mg02_agg_top20 = mg02_agg.head(20)

# 3) Raw MG02 trace_anomalies rows limited to top 20 by timestamp descending with selected columns
if not mg02_trace_anom.empty:
    # Ensure thresholds included (they may already be present); prefer thresholds from trace_thresholds for consistency
    mg02_raw = mg02_trace_anom.merge(
        trace_thresholds[trace_thresholds['cmdb_id']=='MG02'][['trace_name','p95','p5']],
        on='trace_name',
        how='left',
        suffixes=('','_thr')
    )
    mg02_raw_top20 = mg02_raw.sort_values('timestamp', ascending=False)[
        ['timestamp','trace_name','value','p95','p5']
    ].head(20)
else:
    mg02_raw_top20 = pd.DataFrame(columns=['timestamp','trace_name','value','p95','p5'])

# Return compact results: aggregated groups and recent raw anomaly rows
mg02_agg_top20, mg02_raw_top20

```
Out[7]:
```
Summary of MG02 trace anomalies (compact):

- Scope: aggregated MG02 trace_anomalies by trace_name; returned groups sorted by severity_ratio (max / p95).
- Key hotspots (highest severity_ratio first):
  1. trace.to_dockerB2.duration_mean — 4 anomalies, max=2.091s, p95≈0.775s, severity≈2.70 (largest relative latency increase).
  2. trace.from_Tomcat04.duration_mean — 4 anomalies, max=0.8005s, p95≈0.321s, severity≈2.49.
  3. trace.to_dockerA1.duration_mean — 4 anomalies, max=0.4277s, p95≈0.230s, severity≈1.86.
  4. trace.from_dockerB2.duration_mean — 4 anomalies, max=0.1733s, p95≈0.1095s, severity≈1.58.
  5. trace.from_dockerB1.duration_mean — 4 anomalies, max=0.1102s, p95≈0.0783s, severity≈1.41.
- Other notable groups: several to/from MG02 row_count and duration_p95 entries and many duration_mean groups with severity ~1.1–1.3 (moderate increases).
- Timing pattern: many raw MG02 anomaly rows cluster at timestamp 1616562000 (end of the window), indicating a concentrated spike around that minute across multiple trace types and call paths.
- Interpretation and recommended actions:
  - MG02 shows elevated inter-service latencies, most prominently on calls to dockerB2 and replies from Tomcat04 — these call paths should be investigated first.
  - The latency increases are relative to historical baselines (p95), not necessarily huge absolute values, but the relative jumps (≥~2x) for critical paths are meaningful.
  - Correlate these trace spikes with the metric anomalies already observed for MG02 (prolonged TCP FIN-WAIT and an extreme DSKWrite spike) and with logs on dockerB2 / Tomcat04 to identify whether network, disk I/O, or back-end delays caused the trace tail latency.
- Next steps: pull raw spans for the mentioned trace_names around 1616562000, check service logs and connection states, and correlate with MG02 metric_faults for root-cause (disk write, FIN-WAIT, or network congestion).

The original code execution output of IPython Kernel is also provided below for reference:

(                           trace_name  anomaly_count  earliest_timestamp  latest_timestamp  min_value    max_value          p95          p5  severity_ratio
0     trace.to_dockerB2.duration_mean              4            0.059393          2.091000   0.059393     2.091000     0.775405    0.097933        2.696654
1   trace.from_Tomcat04.duration_mean              4            0.065418          0.800500   0.065418     0.800500     0.321282    0.093427        2.491581
2     trace.to_dockerA1.duration_mean              4            0.050492          0.427667   0.050492     0.427667     0.230248    0.072671        1.857415
3   trace.from_dockerB2.duration_mean              4            0.006000          0.173310   0.006000     0.173310     0.109493    0.007493        1.582837
4   trace.from_dockerB1.duration_mean              4            0.006500          0.110175   0.006500     0.110175     0.078291    0.022206        1.407250
5       trace.from_dockerB1.row_count              4            2.000000        376.000000   2.000000   376.000000   281.500000   62.000000        1.335702
6            trace.self.duration_mean              4            0.073220          0.277720   0.073220     0.277720     0.209805    0.085050        1.323702
7   trace.from_dockerA1.duration_mean              4            0.005400          0.146864   0.005400     0.146864     0.114265    0.006112        1.285291
8   trace.from_Tomcat01.duration_mean              4            0.067480          0.363667   0.067480     0.363667     0.285655    0.082577        1.273101
9   trace.from_dockerA2.duration_mean              4            0.006667          0.120821   0.006667     0.120821     0.098109    0.007337        1.231498
10      trace.from_dockerA1.row_count              4            1.000000         40.000000   1.000000    40.000000    32.550000    2.900000        1.228879
11     trace.to_dockerB2.duration_p95              4            0.205000          4.118800   0.205000     4.118800     3.418900    0.648100        1.204715
12   trace.from_Tomcat01.duration_p95              4            0.078500          2.107500   0.078500     2.107500     1.779325    0.455025        1.184438
13   trace.from_Tomcat02.duration_p95              4            0.519050          2.393350   0.519050     2.393350     2.093850    0.885750        1.143038
14  trace.from_Tomcat03.duration_mean              4            0.049020          0.363565   0.049020     0.363565     0.322796    0.091495        1.126301
15      trace.from_dockerB2.row_count              5            7.000000        107.000000   7.000000   107.000000    96.100000   21.000000        1.113424
16      trace.from_dockerA2.row_count              4            2.000000         89.000000   2.000000    89.000000    80.000000    2.500000        1.112500
17          trace.from_root.row_count              2            3.000000         11.000000   3.000000    11.000000     9.950000    3.150000        1.105528
18    trace.to_dockerA2.duration_mean              4            0.066000          0.192675   0.066000     0.192675     0.176471    0.079849        1.091826
19               trace.self.row_count              4           25.000000       1317.000000  25.000000  1317.000000  1215.500000  714.000000        1.083505,       timestamp                         trace_name      value          p95          p5
163  1616562000    trace.to_dockerB2.duration_mean   2.091000     0.775405    0.097933
160  1616562000        trace.to_dockerA2.row_count   6.000000   269.500000  171.000000
159  1616562000    trace.to_dockerA2.duration_mean   0.191167     0.176471    0.079849
162  1616562000        trace.to_dockerB1.row_count   5.000000   288.000000  137.500000
161  1616562000     trace.to_dockerB1.duration_p95   0.827000     3.081375    1.028325
147  1616562000      trace.from_Tomcat03.row_count   7.000000   209.500000  144.500000
148  1616562000  trace.from_Tomcat04.duration_mean   0.800500     0.321282    0.093427
144  1616562000      trace.from_Tomcat01.row_count   3.000000   208.000000  137.500000
143  1616562000  trace.from_Tomcat01.duration_mean   0.363667     0.285655    0.082577
146  1616562000   trace.from_Tomcat03.duration_p95   0.731400     2.093475    0.839850
145  1616562000      trace.from_Tomcat02.row_count   6.000000   205.500000  139.000000
155  1616562000           trace.self.duration_mean   0.277720     0.209805    0.085050
156  1616562000               trace.self.row_count  25.000000  1215.500000  714.000000
157  1616562000    trace.to_dockerA1.duration_mean   0.427667     0.230248    0.072671
158  1616562000        trace.to_dockerA1.row_count   6.000000    94.000000   48.000000
149  1616562000      trace.from_Tomcat04.row_count   4.000000   209.500000  137.000000
150  1616562000  trace.from_dockerA2.duration_mean   0.006667     0.098109    0.007337
153  1616562000   trace.from_dockerB1.duration_p95   0.007850     1.013475    0.017050
154  1616562000      trace.from_dockerB1.row_count   2.000000   281.500000   62.000000
151  1616562000   trace.from_dockerA2.duration_p95   0.007900     1.015775    0.009375)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# timezone (per rule) - not used to convert timestamps but defined for compliance
tz = pytz.timezone('Asia/Shanghai')

# Time window (already in epoch seconds UTC as requested)
start_ts = 1616560200
end_ts = 1616562000

# Components of interest
components = ['MG02','dockerB2','Tomcat04','dockerA1','MG01','Tomcat01','IG01','Redis02']

# Ensure log DataFrame is loaded (reuse if present)
try:
    log
except NameError:
    log = pd.read_csv('log.csv')

# Ensure error_logs DataFrame is loaded
try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')

# 1) Filter log.csv to time window and components
log_window = log[
    (log['timestamp'] >= start_ts) &
    (log['timestamp'] <= end_ts) &
    (log['cmdb_id'].isin(components))
].copy()

# Ensure numeric value
log_window['value'] = pd.to_numeric(log_window['value'], errors='coerce')

# 2) Aggregate per (cmdb_id, log_name)
# total_row_count, min_value, mean_value, max_value, count_nonzero, earliest_nonzero_timestamp
agg_basic = log_window.groupby(['cmdb_id','log_name'])['value'].agg(
    total_row_count = 'count',
    min_value = 'min',
    mean_value = 'mean',
    max_value = 'max'
).reset_index()

# count_nonzero and earliest_nonzero_timestamp
nonzero = log_window[log_window['value'] > 0].copy()
count_nonzero = nonzero.groupby(['cmdb_id','log_name'])['value'].agg(count_nonzero='count').reset_index()
earliest_nonzero = nonzero.groupby(['cmdb_id','log_name'])['timestamp'].agg(earliest_nonzero_timestamp='min').reset_index()

# Merge aggregates
log_agg_window = agg_basic.merge(count_nonzero, on=['cmdb_id','log_name'], how='left') \
                          .merge(earliest_nonzero, on=['cmdb_id','log_name'], how='left')

# Fill NaN for count_nonzero with 0 and keep earliest as NaN where no nonzero
log_agg_window['count_nonzero'] = log_agg_window['count_nonzero'].fillna(0).astype(int)
log_agg_window['earliest_nonzero_timestamp'] = log_agg_window['earliest_nonzero_timestamp'].where(
    ~log_agg_window['earliest_nonzero_timestamp'].isna(), np.nan
)

# 3) Sort and prepare compact output (top 20)
log_agg_window_sorted = log_agg_window.sort_values(['count_nonzero','mean_value'], ascending=[False, False])
log_agg_window_top20 = log_agg_window_sorted[[
    'cmdb_id','log_name','total_row_count','count_nonzero','earliest_nonzero_timestamp',
    'min_value','mean_value','max_value'
]].head(20)

# 4) Filter error_logs to same window and components
error_logs_window = error_logs[
    (error_logs['timestamp'] >= start_ts) &
    (error_logs['timestamp'] <= end_ts) &
    (error_logs['cmdb_id'].isin(components))
].copy()

# Return compact results: aggregated log summary (top 20) and recent error logs (up to 50 rows, sorted desc)
error_logs_window_top50 = error_logs_window.sort_values('timestamp', ascending=False)[['timestamp','cmdb_id','message']].head(50)

# Expose results (variables stored in kernel): log_agg_window, error_logs_window
log_agg_window_top20, error_logs_window_top50

```
Out[8]:
```
```python
summary = (
    "Log window summary (1616560200–1616562000) for selected components:\n"
    "- Tomcat04 & Tomcat01: have full log.row_count series (31 samples each) with mean row_count ~350 per interval (min ~6, max ~439). "
    "All row_count entries are non-zero (count_nonzero=31).\n"
    "- GC pause: Tomcat04 and Tomcat01 show small gc_pause_max values (mean ~0.053–0.055s, max ~0.10s).\n"
    "- log.error_count for both Tomcat hosts is zero across the window (no logged errors in these aggregated metrics).\n"
    "- No error_log entries were found in error_logs.csv for the requested components/time window (error_logs_window is empty).\n\n"
    "Interpretation: Tomcat services show normal traffic volume and small GC pauses during the window; no log-based error signals were found for the selected hosts. "
    "Combine this with the metric and trace anomalies (already extracted) to correlate timing — e.g., MG02 disk/network and trace tail-latency issues observed earlier should be investigated even though Tomcat logs do not show errors."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id          log_name  total_row_count  count_nonzero  earliest_nonzero_timestamp  min_value  mean_value  max_value
5  Tomcat04     log.row_count               31             31                1.616560e+09       6.00  349.870968      439.0
2  Tomcat01     log.row_count               31             31                1.616560e+09       6.00  348.161290      429.0
4  Tomcat04  log.gc_pause_max               19             19                1.616560e+09       0.02    0.053684        0.1
1  Tomcat01  log.gc_pause_max               17             17                1.616560e+09       0.02    0.054706        0.1
0  Tomcat01   log.error_count               31              0                         NaN       0.00    0.000000        0.0
3  Tomcat04   log.error_count               31              0                         NaN       0.00    0.000000        0.0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables: metric, metric_thresholds
tz = pytz.timezone('Asia/Shanghai')  # per rule (no conversion applied)

# Incident window (inclusive)
start_ts = 1616560200
end_ts = 1616562000

# Explicit MG02 candidate KPIs
explicit_kpis = {
    'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT',
    'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite',
    'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps'
}

# Gather MG02 KPIs from metric_thresholds that mention CPU or disk (case-insensitive substrings)
mg02_thresh = metric_thresholds[metric_thresholds['cmdb_id'] == 'MG02']
mg02_kpis_all = mg02_thresh['kpi_name'].dropna().unique()

# Define keywords for CPU and disk (cover common variants)
keywords = ['cpu', 'disk', 'dsk', 'localdisk']

def is_cpu_or_disk(k):
    lk = k.lower()
    return any(kw in lk for kw in keywords)

mg02_cpu_disk_kpis = {k for k in mg02_kpis_all if is_cpu_or_disk(k)}

# Combine explicit KPIs and CPU/disk KPIs
selected_kpis = sorted(set(list(explicit_kpis) + list(mg02_cpu_disk_kpis)))

# Filter metric for MG02, selected KPIs, and time window
mg02_metrics_window = metric[
    (metric['cmdb_id'] == 'MG02') &
    (metric['kpi_name'].isin(selected_kpis)) &
    (metric['timestamp'] >= start_ts) &
    (metric['timestamp'] <= end_ts)
].copy()

# Ensure value numeric
mg02_metrics_window['value'] = pd.to_numeric(mg02_metrics_window['value'], errors='coerce')

# Sort by timestamp ascending
mg02_metrics_window.sort_values('timestamp', inplace=True)

# Compact display: first 200 rows and shape
mg02_metrics_window[['timestamp','cmdb_id','kpi_name','value']].head(200), mg02_metrics_window.shape

```
Out[9]:
```
Summary of MG02 metrics in the incident window:

- The filtered MG02 time series for the window is stored in mg02_metrics_window and contains 561 rows covering CPU, disk, and network KPIs (within 1616560200–1616562000).
- Key types of metrics present: per-CPU SingleCpuidle, CPULoad, CPUWio, and local-disk KPIs (sda/sdb DSKWrite, DSKWTps, DSKRead, DSKRTps), plus other disk/CPU/dsk-related KPIs and the selected FIN-WAIT network KPI.
- Notable sample observations from the extracted rows:
  - At timestamp 1616560200 many per-CPU idle values ~95% (one core showed 0), CPULoad ≈ 1.12 — idle most of the time.
  - CPU I/O wait spike: OSLinux-CPU_CPU_CPUWio had a high sample ≈ 24.316 at 1616560800.
  - Disk activity examples at 1616560860: sda_DSKWrite ≈ 12.933, sda_DSKWTps ≈ 0.85, sda_DSKRead ≈ 7.2, sda_DSKRTps ≈ 0.25.
  - (From prior anomaly/fault results) an extreme sdb_DSKWrite spike (~12,569) was observed at 1616560620 and should be considered a primary concern.
- Actionable next steps:
  - Prioritize investigation of the extreme write spike (≈12,569 at 1616560620) and the sustained FIN-WAIT condition previously detected for MG02.
  - Correlate the CPUWio spikes (e.g., 1616560800) and the disk write/read spikes (1616560620 and 1616560860) with trace anomalies and logs around those timestamps to determine if disk subsystem or I/O contention caused the increased trace latencies.
  - Use mg02_metrics_window for time-series drilldown and correlation with metric_faults, trace_anomalies, and logs.

The original code execution output of IPython Kernel is also provided below for reference:

(       timestamp cmdb_id                                          kpi_name      value
167   1616560200    MG02                   OSLinux-CPU_CPU-0_SingleCpuidle  95.569620
168   1616560200    MG02                   OSLinux-CPU_CPU-1_SingleCpuidle  96.519567
169   1616560200    MG02                   OSLinux-CPU_CPU-2_SingleCpuidle   0.000000
170   1616560200    MG02                   OSLinux-CPU_CPU-3_SingleCpuidle  97.006486
171   1616560200    MG02                           OSLinux-CPU_CPU_CPULoad   1.120000
...          ...     ...                                               ...        ...
8757  1616560800    MG02                            OSLinux-CPU_CPU_CPUWio  24.316008
9663  1616560860    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite  12.933333
9662  1616560860    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps   0.850000
9661  1616560860    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead   7.200000
9660  1616560860    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps   0.250000

[200 rows x 4 columns], (561, 4))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```